# EN/DE Batch Gradient Cosine Similarity

Minimal check in the same spirit as `check_interference_balanced`: sample English and German batches, compute the loss gradient for each language batch separately, then cosine the flattened gradient vectors.

This is **gradient cosine**, not embedding cosine. For the meaningful final-model check, set `CHECKPOINT` to the saved `final_model` path or Hub repo.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoConfig, AutoTokenizer

ROOT = Path("..") if Path("../data/train_lang.csv").exists() else Path(".")
sys.path.insert(0, str(ROOT.resolve()))

from experiments.config import load_config
from experiments.models import SentimentModel

In [ ]:
CONFIG_PATH = ROOT / "configs" / "noise_weight_q5_xlmr_large.json"
TRAIN_CSV = ROOT / "data" / "train_lang.csv"

# Use None for the untrained config-shaped LoRA model, or point this to the final model.
# Example local: ROOT / "outputs" / "..." / "final_model"
# Example Hub: "MichaHenh/cil-noise-weight-q5-xlmr-large-seed1"
CHECKPOINT = None

BATCH_SIZE = 64
NUM_BATCHES = 10
SEED = 42
MAX_LENGTH = 128

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
config = load_config(CONFIG_PATH)
df = pd.read_csv(TRAIN_CSV)

print(df.shape)
print(device)
display(pd.crosstab(df["lang"], df["label"]))

In [ ]:
def load_model_and_tokenizer():
    tokenizer_source = CHECKPOINT or config.model.name
    try:
        tokenizer = AutoTokenizer.from_pretrained(tokenizer_source)
    except (OSError, ValueError):
        tokenizer = AutoTokenizer.from_pretrained(config.model.name)

    if CHECKPOINT is None:
        model = SentimentModel.from_config(config.model, config.objective)
    elif config.model.kind == "lora_bert":
        from peft import PeftModel

        hf_config = AutoConfig.from_pretrained(config.model.name)
        hf_config.num_labels = config.objective.n_classes
        base = SentimentModel.from_pretrained(
            config.model.name,
            config=hf_config,
            geometry=config.model.geometry,
            objective=config.objective,
        )
        model = PeftModel.from_pretrained(base, CHECKPOINT)
    else:
        hf_config = AutoConfig.from_pretrained(config.model.name)
        hf_config.num_labels = config.objective.n_classes
        model = SentimentModel.from_pretrained(
            CHECKPOINT,
            config=hf_config,
            geometry=config.model.geometry,
            objective=config.objective,
        )

    model.to(device)
    model.eval()  # dropout off; gradients still work because we are not using torch.no_grad()
    return model, tokenizer


model, tokenizer = load_model_and_tokenizer()
print("trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
rng = np.random.default_rng(SEED)
labels = sorted(df["label"].unique())
lang_indices = {lang: df.index[df["lang"].eq(lang)].to_numpy() for lang in ["eng_Latn", "deu_Latn"]}
lang_label_indices = {
    (lang, label): df.index[df["lang"].eq(lang) & df["label"].eq(label)].to_numpy()
    for lang in lang_indices
    for label in labels
}


def sample_random(lang, n=BATCH_SIZE):
    pool = lang_indices[lang]
    return rng.choice(pool, size=n, replace=len(pool) < n)


def sample_balanced(lang, n=BATCH_SIZE):
    base = n // len(labels)
    counts = np.full(len(labels), base)
    counts[: n - base * len(labels)] += 1

    parts = []
    for label, count in zip(labels, counts):
        pool = lang_label_indices[(lang, label)]
        parts.append(rng.choice(pool, size=count, replace=len(pool) < count))

    idx = np.concatenate(parts)
    rng.shuffle(idx)
    return idx


def batch_from_indices(indices):
    texts = df.loc[indices, "sentence"].fillna("").astype(str).tolist()
    batch = tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    batch = {key: value.to(device) for key, value in batch.items()}
    batch["labels"] = torch.tensor(df.loc[indices, "label"].to_numpy(), device=device)
    return batch


def grad_for_indices(indices):
    model.zero_grad(set_to_none=True)
    out = model(**batch_from_indices(indices))
    out["loss"].backward()

    grads = []
    for param in model.parameters():
        if param.requires_grad and param.grad is not None:
            grads.append(param.grad.detach().flatten().cpu())
    return torch.cat(grads)


def gradient_cosine(en_idx, de_idx):
    grad_en = grad_for_indices(en_idx)
    grad_de = grad_for_indices(de_idx)
    return F.cosine_similarity(grad_en, grad_de, dim=0).item()


def run_batches(mode, num_batches=NUM_BATCHES):
    sampler = sample_balanced if mode == "balanced" else sample_random
    rows = []
    for batch in range(1, num_batches + 1):
        en_idx = sampler("eng_Latn")
        de_idx = sampler("deu_Latn")
        rows.append(
            {
                "mode": mode,
                "batch": batch,
                "n_eng": len(en_idx),
                "n_deu": len(de_idx),
                "grad_cosine": gradient_cosine(en_idx, de_idx),
            }
        )
        print(rows[-1])
    return pd.DataFrame(rows)

In [ ]:
results = pd.concat(
    [run_batches("random"), run_batches("balanced")],
    ignore_index=True,
)

display(results)
display(results.groupby("mode")["grad_cosine"].agg(["mean", "std", "min", "max"]))